# Tutorial: Custom transforms

The instructions for this tutorial can be found in the [documentation](https://metasmith.readthedocs.io/en/latest/tutorials/custom_transforms.html).

In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, ContainerRuntime
from metasmith.python_api import DataTypeLibrary, DataInstanceLibrary, TransformInstanceLibrary
from metasmith.python_api import Source, Logistics
from metasmith.python_api import TargetBuilder, Resources, Size, Duration
from metasmith.python_api import ipynbButtonLink

WORKSPACE = Path("../../").resolve() # back twice since we are in example_resources/tutorials
MLIB = WORKSPACE/"MetasmithLibraries"
WORKSPACE

In [ ]:
ani_transforms_path = WORKSPACE/"ani_transforms"
ani_transforms = TransformInstanceLibrary(ani_transforms_path)
ani_transforms.AddStub("fastani")
ani_transforms.Save()

In [ ]:
ipynbButtonLink(url=ani_transforms_path/"fastani.py")

In [ ]:
ipynbButtonLink(url=MLIB/"data_types/sequences.yml")

In [ ]:
ani_types_path = WORKSPACE/"ani_types.yml"
ani_types_path.touch()
ipynbButtonLink(url=ani_types_path)

In [ ]:
ani_types = DataTypeLibrary.Load(ani_types_path)
for name, model in ani_types:
    print(name, model)

In [ ]:
ani_transforms_path = WORKSPACE/"ani_transforms"
ani_transforms = TransformInstanceLibrary(ani_transforms_path)
ani_transforms.AddTypeLibrary(lib=ani_types, namespace="ani")   # new
ani_transforms.AddTypeLibrary(MLIB/"data_types/sequences.yml")  # new
ani_transforms.AddTypeLibrary(MLIB/"data_types/pangenome.yml")  # new
ani_transforms.AddStub("fastani")
ani_transforms.Save()

In [ ]:
inputs_path = WORKSPACE/"ani_test_inputs.xgdb"
try:
    inputs = DataInstanceLibrary.Load(inputs_path)
except:
    inputs = DataInstanceLibrary(inputs_path)
    # add data types
    inputs.AddTypeLibrary(MLIB/"data_types/pangenome.yml")
    inputs.AddTypeLibrary(MLIB/"data_types/ncbi.yml")
    inputs.AddTypeLibrary(ani_types, namespace="ani")

    # register inputs
    group = inputs.AddValue("pangenome", "e coli", "pangenome::pangenome")
    inputs.AddValue("DH10b", "GCF_000019425.1", "ncbi::assembly_accession", parents={group})
    inputs.AddValue("K12", "GCF_000005845.2", "ncbi::assembly_accession", parents={group})
    inputs.AddValue("EPI300", "GCF_049667475.1", "ncbi::assembly_accession", parents={group})
    inputs.AddValue("fastani.oci", "docker://staphb/fastani:1.34", "ani::fastani.oci")
    inputs.Save()

In [ ]:
resources = [
    DataInstanceLibrary.Load(MLIB/f"resources/{n}")
    for n in ["containers"]
] + [
    view
    for view in inputs.AsSamples("ani::fastani.oci")
]

transforms = [
    TransformInstanceLibrary.Load(MLIB/f"transforms/{n}")
    for n in ["logistics"]
] + [
    ani_transforms
]

In [ ]:
agent_home = Source.FromLocal(WORKSPACE/"msm_home")
smith = Agent(
    home = agent_home,
    runtime=ContainerRuntime.DOCKER,
)

targets = TargetBuilder()
targets.Add("ani::table")
task = smith.GenerateWorkflow(
    samples=inputs.AsSamples("ncbi::assembly_accession"),
    resources=resources,
    transforms=transforms,
    targets=targets,
)

dag = task.plan.RenderDAG(WORKSPACE/"ani_dag.svg")
ipynbButtonLink(dag)

In [ ]:
smith.StageWorkflow(task, on_exist="update")

smith.RunWorkflow(
    task,
    config_file=smith.GetNxfConfigPresets()["local"],
    params= dict(
        executor=dict(
            cpus=14,
            queueSize=3, # explicitly set 3 jobs to run in parallel
        ),
        process=dict(
            tries=1,
        ),
    ),
    resource_overrides={
        "*": Resources(
            memory=Size.GB(1),
        ),
        "fastani": Resources(
            cpus=14, # give fastANI all the threads
        )
    }
)

In [ ]:
results_path = smith.GetResultSource(task).GetPath()
results = DataInstanceLibrary.Load(results_path)

ipynbButtonLink(results_path/"_metadata/logs.latest/nxf_report.html")

for path, type_name, endpoint in results.Iterate():
    if path.is_absolute(): continue # inputs have absolute paths
    ipynbButtonLink(results_path/path, f'view {type_name} {path.name}')